<a href="https://colab.research.google.com/github/Prktriesitout/Agent/blob/main/PersonaAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# --- State Definition ---
class PersonaState(TypedDict):
    name: str
    designation: str
    query: str
    response: str
# --- Nodes ---
def introduce_agent(state: PersonaState):
    print(f"Hi, I am {state['name']}, your {state['designation']}.")
    return state
# --- Graph Construction ---
graph = StateGraph(PersonaState)
graph.add_node("intro", introduce_agent)
graph.add_edge(START, "intro")
graph.add_edge("intro", END)
# --- Execution ---
app = graph.compile()
app.invoke({"name": "FinBot", "designation": "Personal Finance Advisor", "query": "What are good saving plans?"})


In [ ]:
def handle_usecase(state: PersonaState):
    if "invest" in state["query"].lower():
        state["response"] = "As your finance advisor, I suggest diversified mutual funds."
    else:
        state["response"] = "Let's discuss your financial goals."
    return state
graph.add_node("usecase", handle_usecase)
graph.add_edge("intro", "usecase")
graph.add_edge("usecase", END)
Here, the use case logic is embedded as a LangGraph node, ensuring persona-specific behavior.


In [ ]:
class PersonaConfig(TypedDict):
    name: str
    designation: str
    purpose: str
    problem: str
def describe_purpose(state: PersonaConfig):
    print(f"I am {state['name']} ({state['designation']}). My purpose is {state['purpose']}. I solve: {state['problem']}")
    return state


In [ ]:
Data Sources: Internal DBs, documents, or embeddings.
APIs: For real-time actions (bookings, weather, financial data).
Permissions: Security and access control — what data or actions the agent is authorized for

In [ ]:
Business Example

Agent: LoanEvaluator

Data: Credit history, salary records.
APIs: Bank verification API, credit bureau API.
Customer database
KYC API
Permission to access financial records
Permissions: Read-only access to credit data, write access to loan decisions.

In [ ]:
You can define a resource configuration layer inside the agent’s initialization:
from typing import List
class ResourceState(TypedDict):
    data_sources: List[str]
    apis: List[str]
    permissions: List[str]
def check_resources(state: ResourceState):
    print(f" Data Sources: {state['data_sources']}")
    print(f"APIs: {state['apis']}")
    print(f"Permissions: {state['permissions']}")
    return state


In [ ]:
from langgraph.graph import StateGraph, START, END, subgraph
# --- Sub-agents ---
def planner_node(state):
    state["plan"] = "Collect user preferences and create itinerary."
    return state
def executor_node(state):
    print(f"Executing: {state['plan']}")
    return state
# --- Main Graph ---
main = StateGraph(dict)
main.add_node("planner", planner_node)
main.add_node("executor", executor_node)
main.add_edge(START, "planner")
main.add_edge("planner", "executor")
main.add_edge("executor", END)
app = main.compile()
app.invoke({})


In [ ]:
When to Use What?

Use YAML when:
You want clean, readable configuration files.
You're working with DevOps tools (e.g., Docker, Kubernetes).
You need comments in your config.
Use JSON when:
You’re exchanging data between systems or APIs.
You need strict formatting and universal language support.
You’re working in web development or JavaScript environments.

In [ ]:
agent:
  name: "FinBot"
 designation: "Personal Finance Advisor"
purpose:
 description: "Assist customers with personalized investment and savings strategies."
 problem_statement: "Users struggle to choose the right investment plans aligned with their risk profile."
use_case:
 domain: "Finance"
  type: "Advisory"
 scope:
    - "Provide investment recommendations"
    - "Analyze spending patterns"
    - "Generate savings reports"
resources:
 data_sources:
    - "customer_financial_history_db"
    - "market_trends_api"
  apis:
    - "https://api.stockmarket.com"
    - "https://api.banking.com"
 permissions:
    - "read:financial_data"
    - "write:advice_logs"
interaction:
 human_interaction:
   mode: "Chat-based"
   tone: "Professional and empathetic"
 collaboration:
   partners:
      - "RiskEvalAgent"
      - "PortfolioManagerAgent"
   communication_mode: "Message passing via shared state"
metadata:
 version: "1.0"
 author: "AI Systems Team"
 created_at: "2025-10-16"


introducing personas

In [ ]:
import yaml
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# --- Define Persona State ---
class PersonaState(TypedDict):
   persona: dict
   query: str
   response: str
# --- Load Persona Blueprint ---
with open("persona_blueprint.yaml", "r") as file:
   persona_config = yaml.safe_load(file)
# --- Nodes ---
def introduce_persona(state: PersonaState):
   agent = state["persona"]["agent"]
   print(f" Hello! I am {agent['name']}, your {agent['designation']}.")
   print(f"My purpose: {state['persona']['purpose']['description']}")
   return state
def execute_usecase(state: PersonaState):
   usecase = state["persona"]["use_case"]
    if usecase["domain"].lower() == "finance":
       state["response"] = "Based on your spending, I recommend diversifying into ETFs."
   else:
       state["response"] = "I can help analyze your data."
   print("Response:", state["response"])
   return state
# --- Build Graph ---
graph = StateGraph(PersonaState)
graph.add_node("introduce", introduce_persona)
graph.add_node("usecase", execute_usecase)
graph.add_edge(START, "introduce")
graph.add_edge("introduce", "usecase")
graph.add_edge("usecase", END)
app = graph.compile()
# --- Run ---
app.invoke({"persona": persona_config, "query": "What are the best saving options for 2025?"})


Once an agent’s persona has been modeled — including its identity, purpose, and resources — the next critical step is defining what it can and cannot do.

In human organizations, job roles are effective because employees have:

Clearly defined skills and responsibilities,
Recognized strengths and specializations,
A consistent communication style, and
Awareness of their limitations.
Similarly, an AI agent needs to be designed with explicit boundaries for safe, efficient, and ethical operation.
Understanding an agent’s capabilities and limitations ensures that:

Tasks are routed to the right agents.
End users trust agent responses.
System designers prevent overreach or misuse.
This module prepares you to specify — both conceptually and technically — the operational envelope of your LangGraph-based agentic systems.

Every agent, no matter how sophisticated, is constrained by its design scope, training data, and tool access.  While some agents are optimized for creative reasoning, others excel at structured decision-making or data-driven analysis.  Understanding these boundaries is critical to:

Maintain predictability,
Ensure ethical AI behavior, and
Design interoperable multi-agent ecosystems.

class CapabilityState(TypedDict):
   identity: str
   abilities: list
   query: str
   response: str
def execute_task(state: CapabilityState):
   print(f" Role: {state['identity']}")
    if "analyze" in state["query"].lower():
       state["response"] = "Analyzing financial patterns and risk levels..."
   elif "recommend" in state["query"].lower():
       state["response"] = "Recommending top 3 mutual funds."
   else:
       state["response"] = "I'm not authorized for this task."
   print(state["response"])
   return state


In [ ]:
class SpecializationState(TypedDict):
   strengths: list
   domain: str
   query: str
   response: str
def display_strengths(state: SpecializationState):
   print(f" Domain: {state['domain']}")
   print("Key strengths:")
   for s in state["strengths"]:
       print(f" - {s}")
   return state


In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain.schema import HumanMessage
llm = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=100)
def apply_traits(state):
   tone = "friendly"  # could also be loaded from persona blueprint
   prompt = f"You are a {tone} financial advisor. Respond empathetically to: {state['query']}"
   state["response"] = llm.invoke([HumanMessage(content=prompt)]).content
   print( state["response"])
   return state


In [ ]:
def enforce_limitations(state):
    if "approve loan" in state["query"].lower():
        state["response"] = " I cannot approve loans. I only provide risk assessments."
    return state


In [ ]:
Load Persona (from YAML)
Define Capabilities & Traits
Execute Travel Query
Measure Output Quality
Log Metrics

In [ ]:
from langgraph.graph import StateGraph, START, END
class DecisionState(TypedDict):
    user_input: str
    decision: str
    recommendation: str
def analyze_query(state: DecisionState):
    if "loan" in state["user_input"].lower():
       state["decision"] = "evaluate_risk"
    else:
       state["decision"] = "general_advice"
    return state
def evaluate_risk(state: DecisionState):
   state["recommendation"] = "Based on your profile, you qualify for a medium-risk loan."
    return state
graph = StateGraph(DecisionState)
graph.add_node("analyze_query", analyze_query)
graph.add_node("evaluate_risk", evaluate_risk)
graph.add_edge(START, "analyze_query")
graph.add_conditional_edges("analyze_query", lambda s: s["decision"], {"evaluate_risk": "evaluate_risk"})
graph.add_edge("evaluate_risk", END)
loan_agent = graph.compile()
loan_agent.invoke({"user_input": "I want to apply for a loan"})


In [ ]:
def adapt_contextually(state):
    if "tired" in state["user_input"].lower():
       state["response"] = "It seems you’re exhausted. How about a light workout today?"
    else:
       state["response"] = "Let's continue your fitness plan with a 30-minute run."
    return state


In [ ]:
# Single Agent Architecture
class SingleAgentState(TypedDict):
    task: str
    result: str
def execute_single_task(state):
    if "summarize" in state["task"]:
        state["result"] = "This is a concise summary of your text."
    return state
single_graph = StateGraph(SingleAgentState)
single_graph.add_node("execute", execute_single_task)
single_graph.add_edge(START, "execute")
single_graph.add_edge("execute", END)
single_agent = single_graph.compile()
single_agent.invoke({"task": "summarize"})



In [ ]:
# Multi Agent Orchestration
class MultiAgentState(TypedDict):
    query: str
    result: str
def research_agent(state):
    state["result"] = "Collected 3 relevant sources."
    return state
def summarizer_agent(state):
    state["result"] = "Summarized key insights from sources."
    return state
multi_graph = StateGraph(MultiAgentState)
multi_graph.add_node("research_agent", research_agent)
multi_graph.add_node("summarizer_agent", summarizer_agent)
multi_graph.add_edge(START, "research_agent")
multi_graph.add_edge("research_agent", "summarizer_agent")
multi_graph.add_edge("summarizer_agent", END)
orchestrator = multi_graph.compile()
orchestrator.invoke({"query": "AI trends in 2025"})


Trust Development vs Resource-Oriented Orchestration

Trust Development:
Occurs when agents collaborate through feedback and reinforcement, developing reliability over repeated interactions.

Example:
A LegalAdvisor agent repeatedly validates outputs from a CaseResearcher before reporting.

def validator_agent(state):

    if "invalid" in state["result"]:

        state["feedback"] = "Needs revision"

    else:

        state["feedback"] = "Trusted"

    return state

Resource-Oriented Orchestration:
Focuses on task-to-agent allocation based on available capabilities and efficiency.

Example:
An Orchestrator assigns financial analysis tasks to FinAgent and report generation to DocAgent.

def orchestrator(state):

    if "finance" in state["query"]:

        state["assigned_agent"] = "FinAgent"

    else:

        state["assigned_agent"] = "DocAgent"

    return state

 These two orchestration styles define how multi-agent systems evolve — one via trust, the other via optimization

In [ ]:
system_prompt = """ You are FinBot, a financial advisor. Always provide data-driven, compliant recommendations. """


In [ ]:
examples = [
   {"input": "Best mutual fund?", "output": "Look for diversified funds with low risk."},
   {"input": "High return plan?", "output": "Consider equity-based options, noting higher risk."}
]


In [ ]:
def dynamic_prompt(state):
    context = f"User type: {state['user_profile']}."
    return f"{context} Recommend a financial plan for {state['goal']}."


Data Sources: Raw knowledge inputs from diverse sources - documents (PDFs, manuals, wikis), APIs and tool schemas, ontologies and knowledge graphs, and past conversations.
Ingestion Pipeline: Prepares raw data for structured storage through chunking, metadata enrichment, and embedding creation.
Storage Layers: Organises knowledge for fast retrieval using vector databases, hybrid search indexes, and graph databases.
Retrieval Layer: Fetches relevant knowledge aligned with agent goals through persona-aware retrieval policies, contextual query handling, and multi-step access.
Reasoning Layer: Applies goal-driven reasoning and planning over retrieved knowledge, integrating with LangGraph workflows for orchestration.
Feedback Loop: Enables continuous learning by capturing agent interactions and updating episodic memory and retrieval policies.